# Feature Story 4 — Cluster interpretation

Give business meaning to the three segments found in FS3: who each cluster really is, which real-value thresholds describe it, and a plain-language name we can use with the business.

**Vigilance:** cluster numbers are arbitrary and can shuffle between runs. Both mean tables below are rebuilt from *this* run's data before any threshold or name is written down.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../../data/processed/df_clean_clustered.csv")
cols = ["BALANCE", "PURCHASES", "ONEOFF_PURCHASES",
        "INSTALLMENTS_PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT", "PAYMENTS"]

df.head()


In [ ]:
prep = np.log1p(df[cols])
scaler = StandardScaler()
z = pd.DataFrame(scaler.fit_transform(prep), columns=cols)
z["cluster_final"] = df["cluster_final"].values


### View 1 — standardized means (z-scores)

Which variables are *extreme* per cluster. Positive = above the population average, negative = below. The most extreme values are the ones that tell the story of each segment.

In [ ]:
z_means = z.groupby("cluster_final")[cols].mean().round(2)
print(z_means.to_string())

plt.figure(figsize=(10, 3.5))
sns.heatmap(z_means, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1.5, vmax=1.5, cbar_kws={"label": "z-score"})
plt.title("Standardized means by cluster")
plt.xlabel("")
plt.tight_layout()
plt.show()


### What the z-scores say

**Cluster 0 — Premium Active Spenders.** Everything that means *activity* is above average: purchases (+0.78), one-off purchases (+0.87), credit limit (+0.60) and payments (+0.57). This is the segment that actually uses the card for spending.

**Cluster 1 — Cash-Advance Dependent.** The most extreme row in the table: purchases −1.40 and installments −1.02 (almost nothing bought), but cash advance **+1.00** and balance +0.50. These customers rarely shop with the card — they withdraw cash and carry a balance.

**Cluster 2 — Low-Activity / Low-Balance.** Below average on balance (−0.88), payments (−0.61) and credit limit (−0.56). A quiet segment: small balances, small payments, no cash-advance habit.

In [ ]:
real_means = df.groupby("cluster_final")[cols].mean().round(1)
real_medians = df.groupby("cluster_final")[cols].median().round(1)

print("MEANS (real values)")
print(real_means.to_string())
print()
print("MEDIANS (real values)")
print(real_medians.to_string())


### From profiles to thresholds

The standardized table says *what is extreme*; the real table gives the amounts a business team can act on:

| Cluster | Condition | Why (from the real table) |
|:---|:---|:---|
| 1 | `PURCHASES < 10` | mean $24, median $0 — this segment simply does not buy |
| 0 | `PURCHASES >= 10` and `PAYMENTS > 812` | mean payments $2,889 vs $560 for cluster 2 |
| 2 | `PURCHASES >= 10` and `PAYMENTS <= 812` | the rest: low payments, low balance ($331 mean) |

The next cell checks how well these simplified rules reproduce the KMeans labels. About 81% is expected and fine: KMeans draws spherical boundaries in z-space while these rules are straight axis-aligned cuts. The thresholds exist to *communicate* the segments, not to re-run the clustering.

In [ ]:
rules = pd.Series(np.where(df["PURCHASES"] < 10, 1,
                 np.where(df["PAYMENTS"] > 812, 0, 2)))

print("agreement with cluster_final:", f"{(rules == df['cluster_final']).mean():.1%}")
print()
for k in (0, 1, 2):
    tp = ((rules == k) & (df["cluster_final"] == k)).sum()
    fp = ((rules == k) & (df["cluster_final"] != k)).sum()
    fn = ((rules != k) & (df["cluster_final"] == k)).sum()
    print(f"cluster {k}: precision {tp / (tp + fp):.3f}   recall {tp / (tp + fn):.3f}")


### Segment names

| # | Name | Profile |
|:--|:-----|:--------|
| 0 | **Premium Active Spenders** | High spenders with high limits ($6,526 average). They run $2,199 of purchases on average, mostly one-off, and pay $2,889 back. The natural segment for credit-limit increases and rewards offers. |
| 1 | **Cash-Advance Dependent** | Almost no purchase activity ($24 average, median $0) but $2,009 of cash advances and the highest balances. Spending on the card has stopped; cash is the product. A monitoring / retention-risk segment. |
| 2 | **Low-Activity / Low-Balance** | Small accounts: $331 balance, $560 payments, $2,688 limit. Some everyday purchases ($467) but nothing extreme. A dormant segment — candidates for reactivation campaigns. |

### The final `target` column

Map every cluster number to its name, then run the three required checks: no cluster forgotten, no missing value in `target`, and proportions identical to the original clusters.

In [ ]:
names = {
    0: "Premium Active Spenders",
    1: "Cash-Advance Dependent",
    2: "Low-Activity / Low-Balance",
}
df["target"] = df["cluster_final"].map(names)

assert set(df["cluster_final"].unique()) == set(names), "every cluster needs a name"
assert df["target"].notna().all(), "no missing value in target"

sizes = (df.groupby(["cluster_final", "target"]).size()
           .rename("n").reset_index())
sizes["share"] = (sizes["n"] / sizes["n"].sum() * 100).round(1)

original = df["cluster_final"].value_counts(normalize=True).sort_index()
assert np.allclose(sizes["n"] / sizes["n"].sum(), original.values), "proportions changed"

print(sizes.to_string(index=False))


In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=sizes, x="target", y="n", hue="target",
                 legend=False, palette="Set2")
for patch, n, share in zip(ax.patches, sizes["n"], sizes["share"]):
    ax.annotate(f"{n:,} ({share}%)",
                (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.title("Customers per segment")
plt.xlabel("")
plt.ylabel("customers")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
key = ["PURCHASES", "CASH_ADVANCE", "PAYMENTS"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, v in zip(axes, key):
    sns.barplot(data=df, x="target", y=v, hue="target",
                legend=False, palette="Set2", ax=ax)
    ax.set_title(v)
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=15)
fig.suptitle("Real values by segment")
plt.tight_layout()
plt.show()


In [ ]:
df.to_csv("../../data/processed/df_clean_clustered.csv", index=False)
print("saved:", df.shape)
print("columns:", list(df.columns))


### Findings

- Three balanced segments: **3,393 / 2,428 / 3,128** customers (37.9% / 27.1% / 35.0%) — no fragment, no giant blob.
- The two-table method (z-score view for *which* variable, real view for *how much*) produced thresholds that reproduce ~81% of the KMeans labels — good enough to describe the segments in business words.
- `target` was created with every cluster mapped, 0 missing values and proportions identical to `cluster_final`.
- **FS5 starts from this file**: supervised models learn to predict `target` from the 7 raw variables.